# Module B3: Hybrid Chatbot System

This notebook demonstrates building a hybrid FAQ chatbot. It uses exact/keyword matching for rules, trains an ML classifier (TF-IDF + Logistic Regression) on FAQ patterns, and combines them to form a robust responder with a fallback mechanism.

In [ ]:
import json
import numpy as np
import os
import pickle
import random
import sys
sys.path.append('../')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from app.services.nlp_service import preprocess_text

print('Libraries imported successfully.')

## 1. Load FAQ Intents Dataset

In [ ]:
intents_path = '../data/intents.json'
with open(intents_path, 'r') as f:
    intents_data = json.load(f)

print(f"Loaded {len(intents_data['intents'])} intents.")

## 2. Preprocess Training Data

In [ ]:
X = []
y = []

for intent in intents_data['intents']:
    tag = intent['tag']
    for pattern in intent['patterns']:
        cleaned = preprocess_text(pattern)
        X.append(cleaned)
        y.append(tag)

print(f'Training dataset size: {len(X)} patterns')
print('Sample pattern:', X[0], '->', y[0])

## 3. Train Classifier

In [ ]:
vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(X)

model = LogisticRegression()
model.fit(X_vec, y)

print('Chatbot classifier trained successfully.')

## 4. Build Hybrid Chatbot Responder

In [ ]:
# Rule-based mapping for strict/exact keyword matches
strict_rules = {
    'hi': 'greeting',
    'hello': 'greeting',
    'bye': 'goodbye',
    'hours': 'store_hours',
    'address': 'location',
    'refund': 'refund_status',
    'support': 'contact_support'
}

def get_bot_response(user_query, threshold=0.35):
    # Rule check
    query_lower = user_query.lower().strip()
    for keyword, tag in strict_rules.items():
        if keyword in query_lower:
            # Find the intent
            for intent in intents_data['intents']:
                if intent['tag'] == tag:
                    return random.choice(intent['responses']), tag, 1.0
                    
    # ML fall back
    cleaned = preprocess_text(user_query)
    vec = vectorizer.transform([cleaned])
    pred_tag = model.predict(vec)[0]
    probs = model.predict_proba(vec)[0]
    class_idx = list(model.classes_).index(pred_tag)
    conf = probs[class_idx]
    
    if conf >= threshold:
        for intent in intents_data['intents']:
            if intent['tag'] == pred_tag:
                return random.choice(intent['responses']), pred_tag, conf
                
    return "I am sorry, I didn't quite get that. Could you please rephrase or contact support at support@smartretail.com?", "fallback", conf

# Test chatbot responder
query = 'when does the store open?'
response, tag, conf = get_bot_response(query)
print(f"Query: '{query}'\nResponse: {response}\nTag: {tag} (confidence: {conf:.4f})")

## 5. Save Model and Vectorizer

In [ ]:
os.makedirs('../app/models', exist_ok=True)
chatbot_model = {
    'model': model,
    'vectorizer': vectorizer,
    'intents': intents_data['intents'],
    'strict_rules': strict_rules
}

with open('../app/models/chatbot_model.pkl', 'wb') as f:
    pickle.dump(chatbot_model, f)
print('Chatbot model serialized.')